# 🧭 Bridgeway – Custom ReAct Agent with Configurable Personas

**EECE 503P – C4: Build a Custom ReAct Agent with Configurable Personas**

This notebook continues the C3 project (*Bridgeway – Fundraising & Communications Consultancy for NGOs*, Hamra, Beirut).
In C3 the assistant received the whole business text in its prompt and used the OpenAI tool-calling API.
In C4 the assistant becomes a **ReAct agent**: it *reasons* about what it needs, *acts* by calling a tool,
reads the *observation*, and repeats until it can give a final answer.

**ReAct in one line:** `Thought → Action → Observation → … → Final Answer`

**How this notebook is built (requirements):**
- **LangGraph only.** No `AgentExecutor`, no `create_react_agent`, no prebuilt `ToolNode`. The loop is written by hand:
  an `agent` node, a `tools` node and a conditional edge you can read in section 5.
- Same LLM provider and `.env` key as C3 (`llm.py`: Gemini through its OpenAI-compatible endpoint, or OpenAI). The key is never printed.
- Our own logic detects when the model asks for an action, runs the tool, returns the observation and loops until a final answer.
  Every Thought / Action / Observation is printed.
- Tools: `search_business_info`, `record_customer_interest`, `record_feedback`, `estimate_campaign_package` (`tools.py`).
  Their descriptions are injected into the system prompt.
- Two personas (Layla, Dr. Karam), each with prompt traits **and** a matching LLM configuration.
- A separate, labeled **Instructions** block; logged experiments with a comparison table; a reflection section; a Gradio demo.

| Section | What it does |
|---|---|
| 1. Setup | imports, LLM client from `.env` |
| 2. Tools | the tools and how their descriptions reach the prompt |
| 3. Prompt format | how we ask the model to "think out loud" in the ReAct format |
| 4. Parser | our own code that detects *Action* vs *Final Answer* in the model's text |
| 5. LangGraph loop | `agent` node → conditional edge → `tools` node → back to `agent` |
| 6. First run | the loop in action with `search_business_info` |
| 7. All four tools | `record_customer_interest`, `record_feedback`, `estimate_campaign_package` switched on and triggered |
| 8. Personas | Layla and Dr. Karam: prompt traits + LLM config with justifications, side-by-side runs |
| 9. Instructions | the separate, labeled INSTRUCTIONS block (numbered rules, I/O format, worked example) wired into both personas |
| 10. Experiments | 5 fixed questions × 2 personas × 4 experiments, logged to `logs/experiments.csv`, with a comparison table |

## 1. Setup

`llm.py` is reused unchanged from C3: it reads `LLM_PROVIDER` and the API key from `.env` and returns an
OpenAI-style client. If the key is missing you get a clear message instead of a crash.

In [ ]:
import json
import re
from typing import Optional, TypedDict

import openai
import pandas as pd
from langgraph.graph import END, START, StateGraph

from knowledge import load_business_summary
from llm import LLMConfigError, create_client, get_llm_settings
from log_utils import safe_print
from tools import MCP_TOOL_DEFINITIONS, run_tool

try:
    client, MODEL = create_client()
    PROVIDER = get_llm_settings()[0]          # "gemini" or "openai" (some settings depend on it)
    print(f"✅ LLM client ready (provider: {PROVIDER}, model: {MODEL})")  # the key itself is never printed
except LLMConfigError as e:
    client, MODEL, PROVIDER = None, None, None
    print("⚠️", e)

## 2. 🛠️ Tools

The tools live in `tools.py` (reused from C3, extended for C4). Each one is described MCP-style
(`name`, `description`, `inputSchema`) and the same definitions are used **twice**:

1. `describe_tools()` turns them into text for the system prompt, so the model knows what it may call and how.
2. `run_tool(name, arguments)` executes the tool the model asked for and *never raises*: a wrong tool name,
   broken JSON or missing arguments become `{"status": "error", ...}` that the model can read and fix.

In this first step only **`search_business_info(query)`** is active. It is a small keyword search over
`me/business_summary.txt` and `me/about_business.pdf`. Unlike C3, the business facts are *not* pasted into
the prompt any more, so the agent must decide to search before it answers.

In [ ]:
# Which tools the agent may use right now (more are switched on in later steps)
ACTIVE_TOOL_NAMES = ["search_business_info"]


def tool_definitions(names=None):
    """The MCP-style definitions of the active tools (all tools if names is None)."""
    names = names or ACTIVE_TOOL_NAMES
    return [t for t in MCP_TOOL_DEFINITIONS if t["name"] in names]


def describe_tools(definitions):
    """Render tool definitions as text for the system prompt (name, purpose, JSON arguments)."""
    blocks = []
    for tool in definitions:
        props = tool["inputSchema"]["properties"]
        args = ", ".join(f'"{name}": <{spec.get("type", "string")}: {spec.get("description", "")}>'
                         for name, spec in props.items())
        blocks.append(f"### {tool['name']}\n{tool['description']}\nAction Input: {{{args}}}")
    return "\n\n".join(blocks)


ACTIVE_TOOLS = tool_definitions()
print(describe_tools(ACTIVE_TOOLS))

In [ ]:
# The tool on its own, before any model is involved
print(run_tool("search_business_info", {"query": "donor outreach campaign price"}))
print(run_tool("search_business_info", {"query": "bitcoin donations"}))   # -> no_match
print(run_tool("search_business_info", "{not json"))                     # -> error, never a crash

## 3. 📝 The ReAct prompt format

We do **not** use the API's built-in function calling. Instead the system prompt teaches the model a small
text protocol, and in section 4 we parse that text ourselves. This keeps every step visible:

```
Thought: I need the price of a donor campaign.
Action: search_business_info
Action Input: {"query": "donor outreach campaign price"}
```
→ we run the tool and send back `Observation: {...}` → the model continues →
```
Thought: The campaign costs 600 USD; the small-NGO discount is 20%.
Final Answer: A donor outreach campaign costs 480 USD for a small NGO ...
```

The call uses `stop=["Observation:"]` so the model can never *invent* an observation: generation stops the
moment it tries to write one, and the real tool result is inserted instead.

`build_system_prompt()` assembles: **persona** + **instructions** + **tool descriptions** + **format rules**.
For this step the persona is a plain placeholder; Layla and Dr. Karam arrive in the next step.

In [ ]:
REACT_FORMAT = """## How to answer (ReAct format)
You work in small steps. Every reply MUST have exactly one of these two shapes.

Shape A - you need information or must record something:
Thought: <what you know so far and which tool you need, in one or two sentences>
Action: <tool name, exactly as listed above>
Action Input: <a JSON object with the tool's arguments>
Then STOP. The system runs the tool and replies with a line starting with "Observation:". Never write an Observation yourself.

Shape B - you can answer the user now:
Thought: <short reasoning>
Final Answer: <your reply to the user, in character>

Rules: one Action per reply. Use only the listed tools. Never guess prices, discounts or facts: search first, then answer from the Observation."""

# Temporary persona for step 1 (replaced by Layla / Dr. Karam in step 2)
BASIC_PERSONA = """## Persona
You are the assistant of Bridgeway - Fundraising & Communications Consultancy for NGOs, Hamra, Beirut.
Be helpful, concise and honest; say so when the documents do not contain the answer."""


def build_system_prompt(persona, instructions, tools=None, react_format=None):
    """persona + instructions + tool descriptions + ReAct format -> one system prompt."""
    tools = tools or ACTIVE_TOOLS
    parts = [persona.strip()]
    if instructions and instructions.strip():
        parts.append(instructions.strip())
    parts.append("## Tools you can call\n" + describe_tools(tools))
    parts.append(react_format or REACT_FORMAT)     # the experiments swap in a no-Thought variant
    return "\n\n".join(parts)


print(build_system_prompt(BASIC_PERSONA, ""))

## 4. 🔍 Detecting what the model wants (our own parser)

The model answers with plain text. `parse_model_output()` reads it and decides:

| It finds | Result |
|---|---|
| `Action:` + `Action Input: {json}` | `{"type": "action", "tool": ..., "input": {...}}` → run the tool |
| `Final Answer:` | `{"type": "final", "answer": ...}` → stop |
| plain text with no markers | treated as a final answer, flagged `format_ok=False` (useful for the experiments) |
| anything else (e.g. a Thought with nothing after it, broken JSON) | `{"type": "malformed"}` → the loop asks the model to try again |

It also extracts the `Thought:` so we can print and log it.

In [ ]:
THOUGHT_RE = re.compile(r"Thought:\s*(.*?)(?=\n\s*(?:Action|Action Input|Final Answer)\s*:|\Z)", re.I | re.S)
ACTION_RE = re.compile(r"Action:\s*`?([A-Za-z_]\w*)`?", re.I)
FINAL_RE = re.compile(r"Final Answer:\s*(.*)", re.I | re.S)
# JSON object after "Action Input:", optionally wrapped in ```json fences
INPUT_RE = re.compile(r"Action Input:\s*(?:```(?:json)?\s*)?(\{.*\})", re.I | re.S)


def parse_model_output(text):
    """Turn the model's text into a dict: type = action | final | malformed."""
    text = (text or "").strip()
    thought_match = THOUGHT_RE.search(text)
    thought = thought_match.group(1).strip() if thought_match else ""
    action_match = ACTION_RE.search(text)
    final_match = FINAL_RE.search(text)

    # An Action that comes before any "Final Answer" wins: the model wants a tool.
    if action_match and (final_match is None or action_match.start() < final_match.start()):
        input_match = INPUT_RE.search(text, action_match.end())
        raw = input_match.group(1).strip() if input_match else "{}"
        try:
            arguments = json.loads(raw)
        except json.JSONDecodeError:
            # Greedy match may have swallowed trailing text: retry with the shortest {...}
            shortest = re.search(r"\{.*?\}", raw, re.S)
            try:
                arguments = json.loads(shortest.group(0)) if shortest else None
            except json.JSONDecodeError:
                arguments = None
            if arguments is None:
                return {"type": "malformed", "thought": thought, "reason": f"Action Input is not valid JSON: {raw[:80]!r}"}
        return {"type": "action", "thought": thought, "tool": action_match.group(1), "input": arguments}

    if final_match:
        return {"type": "final", "thought": thought, "answer": final_match.group(1).strip(), "format_ok": True}

    if text and not thought:
        # No markers at all: the model just talked. Accept it, but remember it skipped the format.
        return {"type": "final", "thought": "", "answer": text, "format_ok": False}

    return {"type": "malformed", "thought": thought, "reason": "no Action or Final Answer found"}


# Quick self-test of the parser (no model needed)
samples = [
    'Thought: I need the price.\nAction: search_business_info\nAction Input: {"query": "donor campaign price"}',
    "Thought: The documents say 600 USD minus 20%.\nFinal Answer: It costs 480 USD.",
    "Hello! How can I help you today?",
    "Thought: Hmm, let me think about this.",
    'Action: search_business_info\nAction Input: ```json\n{"query": "team"}\n```',
]
for s in samples:
    print(parse_model_output(s))

## 5. 🔁 The ReAct loop in LangGraph (hand-written)

A LangGraph graph is a **state** (a dictionary) plus **nodes** (functions that read the state and return the
fields they changed) plus **edges** (which node runs next). Ours:

```
START ──► agent ──(should_continue)──┬──► tools ──► agent   (pending_action found)
                                     ├──► agent           (malformed reply → retry)
                                     └──► END             (final_answer found)
```

- **`agent` node** – calls the LLM, prints the *Thought*, and uses `parse_model_output()` to detect an *Action*
  or a *Final Answer*.
- **`tools` node** – runs the requested tool with `run_tool()`, prints the *Observation*, and appends it to the
  conversation so the model sees it on the next turn.
- **`should_continue`** – the conditional edge: `"tools"`, `"retry"` or `"end"`.
- `MAX_STEPS` stops a runaway loop: after that many model calls the agent is told to answer with what it has.

`llm_config` (temperature, top_p, …) travels inside the state, so the **same compiled graph** can run with any
persona or experiment configuration later. Two details about how the config reaches the API:

- **Hidden thinking is switched off.** `gemini-2.5-flash` normally "thinks" silently before answering, and those
  hidden tokens count against `max_tokens`. `thinking_budget: 0` disables that, so the visible `Thought:` line is the
  *only* chain of thought and a short `max_tokens` is not eaten up. (Sent as Gemini's `thinking_config`; ignored for OpenAI.)
- **Provider-specific settings are tried, then dropped if refused.** `top_k`, `frequency_penalty` and
  `presence_penalty` are not accepted by every endpoint. `call_llm` sends them, and if the provider answers
  *400 Unknown name "…"*, it remembers the setting in `UNSUPPORTED_PARAMS`, prints a note and retries without it.

In [ ]:
MAX_STEPS = 6           # max LLM calls per question (keeps a confused model from looping forever)
VERBOSE = True          # print Thought / Action / Observation lines (switched off during experiments)

DEFAULT_LLM_CONFIG = {"temperature": 0.2, "max_tokens": 600, "thinking_budget": 0}
FALLBACK_ANSWER = "Sorry, I could not complete this request. Please contact the team at hello@bridgeway.lb."


def log(line):
    if VERBOSE:
        safe_print(line)


class AgentState(TypedDict):
    messages: list                   # chat history sent to the LLM (system, user, assistant, observations)
    llm_config: dict                 # sampling settings for this run (temperature, top_p, max_tokens, ...)
    pending_action: Optional[dict]   # {"tool", "input"} detected by agent_node, consumed by tools_node
    final_answer: Optional[str]      # set by agent_node when the model gives a Final Answer
    steps: int                       # number of LLM calls so far
    trace: list                      # Thought / Action / Observation records (for printing + experiments)


# Settings the OpenAI-style API accepts as normal arguments. Everything else in a config is
# provider-specific and goes into extra_body:
#   top_k           -> Gemini-only sampling setting
#   thinking_budget -> Gemini's hidden reasoning budget (0 = off, so Thought: is the only chain of thought)
# If the provider rejects a setting, call_llm remembers it in UNSUPPORTED_PARAMS and retries without it.
STANDARD_PARAMS = {"temperature", "top_p", "max_tokens", "frequency_penalty", "presence_penalty"}
UNSUPPORTED_PARAMS = set()


def make_api_kwargs(llm_config):
    """Split a config dict into normal API arguments + extra_body for provider-specific ones."""
    config = {k: v for k, v in llm_config.items() if k not in UNSUPPORTED_PARAMS}
    kwargs = {k: v for k, v in config.items() if k in STANDARD_PARAMS}
    extra = {k: v for k, v in config.items() if k not in STANDARD_PARAMS and k != "thinking_budget"}
    if "thinking_budget" in config and PROVIDER == "gemini":
        # The shape Gemini's OpenAI-compatible endpoint expects for its own settings
        extra["extra_body"] = {"google": {"thinking_config": {"thinking_budget": config["thinking_budget"]}}}
    if extra:
        kwargs["extra_body"] = extra
    return kwargs


def call_llm(messages, llm_config):
    """One chat-completion call. stop=["Observation:"] prevents the model from inventing tool results."""
    if client is None:
        raise LLMConfigError("No LLM client: add your API key to .env and re-run the Setup cell.")
    for _ in range(len(llm_config) + 1):
        kwargs = make_api_kwargs(llm_config)
        try:
            response = client.chat.completions.create(
                model=MODEL, messages=messages, stop=["Observation:"], **kwargs
            )
            return response.choices[0].message.content or ""
        except openai.BadRequestError as e:
            # The provider refused a setting: find which one, remember it, retry without it.
            message = str(e)
            rejected = [k for k in llm_config if k not in UNSUPPORTED_PARAMS
                        and (k in message or (k == "thinking_budget" and "thinking" in message))]
            if not rejected:   # the error does not name a setting: drop every non-standard one
                rejected = [k for k in llm_config if k not in STANDARD_PARAMS and k not in UNSUPPORTED_PARAMS]
            if not rejected:
                raise
            UNSUPPORTED_PARAMS.update(rejected)
            safe_print(f"ℹ️ This provider rejected {rejected}; continuing without it (see UNSUPPORTED_PARAMS).")
    raise RuntimeError("No parameter combination was accepted by the provider.")


# ---- Node 1: the agent (Thought + decision) -------------------------------------------
def agent_node(state: AgentState):
    steps = state["steps"] + 1
    messages = list(state["messages"])
    trace = list(state["trace"])
    out_of_steps = steps > MAX_STEPS

    if out_of_steps:
        messages.append({"role": "user", "content": "You have used all your steps. Reply now with "
                                                    "'Final Answer:' using what you already know."})

    text = call_llm(messages, state["llm_config"])
    messages.append({"role": "assistant", "content": text})
    parsed = parse_model_output(text)

    if parsed["thought"]:
        log(f"💭 Thought: {parsed['thought']}")
        trace.append({"step": steps, "type": "thought", "content": parsed["thought"]})

    # Case A: the model wants a tool -> hand it to the tools node via pending_action
    if parsed["type"] == "action" and not out_of_steps:
        log(f"🔧 Action: {parsed['tool']}({json.dumps(parsed['input'], ensure_ascii=False)})")
        trace.append({"step": steps, "type": "action", "tool": parsed["tool"], "input": parsed["input"]})
        return {"messages": messages, "steps": steps, "trace": trace,
                "pending_action": {"tool": parsed["tool"], "input": parsed["input"]}, "final_answer": None}

    # Case B: final answer (or we force one because the step budget is spent)
    if parsed["type"] == "final" or out_of_steps:
        answer = parsed.get("answer") or (text.strip() if parsed["type"] != "action" else "") or FALLBACK_ANSWER
        log(f"✅ Final Answer: {answer}")
        trace.append({"step": steps, "type": "final", "content": answer, "format_ok": parsed.get("format_ok", False)})
        return {"messages": messages, "steps": steps, "trace": trace, "pending_action": None, "final_answer": answer}

    # Case C: malformed -> tell the model and let the conditional edge send us back here
    log(f"⚠️ Unparseable reply ({parsed['reason']}); asking the model to retry")
    trace.append({"step": steps, "type": "malformed", "content": text})
    messages.append({"role": "user", "content": "Your reply did not follow the format. Reply again with either "
                                                "'Action:' + 'Action Input:' or 'Final Answer:'."})
    return {"messages": messages, "steps": steps, "trace": trace, "pending_action": None, "final_answer": None}


# ---- Node 2: the tools (Action -> Observation) -------------------------------------------
def tools_node(state: AgentState):
    action = state["pending_action"]
    result = run_tool(action["tool"], action["input"])          # never raises
    observation = json.dumps(result, ensure_ascii=False)
    log(f"👁️ Observation: {observation}")
    messages = list(state["messages"]) + [{"role": "user", "content": f"Observation: {observation}"}]
    trace = list(state["trace"]) + [{"step": state["steps"], "type": "observation",
                                     "tool": action["tool"], "result": result}]
    return {"messages": messages, "trace": trace, "pending_action": None}


# ---- The conditional edge -------------------------------------------------------------
def should_continue(state: AgentState):
    if state["pending_action"] is not None:
        return "tools"      # Action detected -> run it
    if state["final_answer"] is not None:
        return "end"        # Final Answer detected -> stop
    return "retry"          # malformed reply -> call the model again


# ---- Assemble the graph ---------------------------------------------------------------
graph = StateGraph(AgentState)
graph.add_node("agent", agent_node)
graph.add_node("tools", tools_node)
graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", should_continue, {"tools": "tools", "retry": "agent", "end": END})
graph.add_edge("tools", "agent")
react_agent = graph.compile()

print(react_agent.get_graph().draw_mermaid())

## 6. ▶️ First run

`run_react()` builds the starting state (system prompt + optional history + the question), invokes the graph and
returns the final answer together with the full trace. Watch the printed `Thought → Action → Observation → Final Answer`.

In [ ]:
def run_react(question, system_prompt=None, llm_config=None, history=None):
    """Ask the agent one question. Returns {"answer", "trace", "steps", "messages", "error"}."""
    system_prompt = system_prompt or build_system_prompt(BASIC_PERSONA, "")
    state: AgentState = {
        "messages": [{"role": "system", "content": system_prompt}, *(history or []),
                     {"role": "user", "content": question}],
        "llm_config": llm_config or DEFAULT_LLM_CONFIG,
        "pending_action": None, "final_answer": None, "steps": 0, "trace": [],
    }
    log(f"\n🧑 User: {question}\n" + "-" * 70)
    try:
        final_state = react_agent.invoke(state)
    except Exception as e:  # API errors, missing key, network ... -> a message, not a crash
        safe_print(f"⚠️ Agent error: {type(e).__name__}: {e}")
        return {"answer": None, "trace": state["trace"], "steps": state["steps"],
                "messages": state["messages"], "error": f"{type(e).__name__}: {e}"}
    return {"answer": final_state["final_answer"], "trace": final_state["trace"],
            "steps": final_state["steps"], "messages": final_state["messages"], "error": None}


result = run_react("How much is a donor campaign for a small NGO?")

In [ ]:
result = run_react("Who is on the Bridgeway team and when are you open?")

In [ ]:
# A question the documents cannot answer: the agent should search, see no_match, and say it doesn't know.
result = run_react("Do you accept crypto donations on behalf of NGOs?")

## 7. 🧰 All four tools

Now the agent gets its full toolbox (all in `tools.py`):

| Tool | When the model should call it | What it does |
|---|---|---|
| `search_business_info(query)` | any factual question about Bridgeway | keyword search in the business documents |
| `record_customer_interest(email, name, message)` | someone leaves a name + email and wants a service | validates the input and appends the lead to `logs/leads.jsonl` |
| `record_feedback(question)` | a question the documents can't answer, a service we don't offer, praise or complaints | appends to `logs/feedback.jsonl` |
| `estimate_campaign_package(ngo_size, goal)` **(new)** | "how much…", "what would it cost…", quotes, budgets | maps the goal to services and computes the price with the one applicable discount |

`estimate_campaign_package` takes its numbers from `business_data.py` — the same prices and discount rules
printed in `me/business_summary.txt` and the PDF — so the model never does arithmetic itself and cannot
invent a price. Switching the tools on is just a list change: their descriptions flow into the system prompt
through `describe_tools()` and `run_tool()` already knows how to run them.

In [ ]:
ACTIVE_TOOL_NAMES = ["search_business_info", "record_customer_interest", "record_feedback", "estimate_campaign_package"]
ACTIVE_TOOLS = tool_definitions()
print(describe_tools(ACTIVE_TOOLS))

In [ ]:
# The estimate tool on its own (no model): the 480 USD example from the documents, a bundle, and two errors
print(run_tool("estimate_campaign_package", {"ngo_size": "small", "goal": "reach donors"}))
print(run_tool("estimate_campaign_package", {"ngo_size": "large", "goal": "donors and volunteers"}))
print(run_tool("estimate_campaign_package", {"ngo_size": "huge", "goal": "donors"}))
print(run_tool("estimate_campaign_package", {"ngo_size": "small", "goal": "paid ads"}))

One question per tool. Watch which **Action** the model picks and what comes back as the **Observation**.

In [ ]:
# -> record_customer_interest
result = run_react("Hi, I'm Maya from GreenSteps (maya@greensteps.org). We need volunteers in March - can you help?")

In [ ]:
# -> record_feedback (a service Bridgeway does not offer)
result = run_react("Can you write our EU grant proposal for us?")

In [ ]:
# -> estimate_campaign_package
result = run_react("We're a small NGO with a 60,000 USD budget. What would it cost to reach new donors and recruit volunteers?")

In [ ]:
# What the tools wrote to disk (both files are gitignored)
from pathlib import Path
for name in ("leads.jsonl", "feedback.jsonl"):
    path = Path("logs") / name
    print(f"--- {path}:")
    print(path.read_text(encoding="utf-8").strip().splitlines()[-1] if path.exists() else "(empty)")

## 8. 🎭 Personas: Layla and Dr. Karam

A persona here is **two things that must match**:

1. **Prompt traits** – tone, vocabulary, response style and values, written into the system prompt.
2. **An LLM configuration** – how the model *samples* words:
   - `temperature` – how random the next word is (0 = always the most likely word, 1 = adventurous).
   - `top_p` – only choose among the most likely words whose probabilities add up to *p*.
   - `top_k` – only choose among the *k* most likely words (a Gemini setting, sent through `extra_body`).
   - `max_tokens` – the maximum length of one reply.
   - `frequency_penalty` / `presence_penalty` – discourage repeating words / encourage new topics.
   - `thinking_budget` – Gemini's hidden reasoning; `0` keeps the visible `Thought:` as the only chain of thought.

A warm advisor who writes with `temperature=0` sounds robotic; a compliance expert with `temperature=0.9`
starts improvising figures. So each setting below carries a one-line justification tied to the persona.
Settings the provider does not accept (`top_k` and the penalties on Gemini's OpenAI-compatible endpoint) are
dropped automatically by `call_llm` and listed in `UNSUPPORTED_PARAMS` after the first run.

In [ ]:
PERSONAS = {
    "Layla": {
        "role": "Warm partnerships advisor",
        "prompt": """## Persona: Layla - Partnerships Advisor at Bridgeway
- Tone: warm, encouraging and personal. You sound like a trusted colleague who genuinely admires the NGO's work; use the person's name when you know it.
- Vocabulary: everyday, friendly words ("together", "partners", "community", "happy to help"); no jargon, no legal language.
- Response style: short paragraphs of 2-5 sentences, at most one emoji, one gentle follow-up question, and a soft next step (the free 30-minute discovery call) when it fits.
- Values: relationships over transactions; respect for small budgets; honesty (never promise how much money a campaign will raise); never pushy.""",
        "llm_config": {"temperature": 0.8, "top_p": 0.95, "top_k": 50, "max_tokens": 800,
                       "frequency_penalty": 0.3, "presence_penalty": 0.2, "thinking_budget": 0},
        "why": {
            "temperature": "0.8 - varied, natural phrasing so the warmth does not sound scripted",
            "top_p": "0.95 - a wide range of word choices for a conversational, human tone",
            "top_k": "50 - allows variety while still filtering out odd or unlikely words",
            "max_tokens": "800 - room for a friendly follow-up question and a next step",
            "frequency_penalty": "0.3 - discourages repeating the same friendly phrases",
            "presence_penalty": "0.2 - nudges her to bring in something new (e.g. a next step) instead of circling",
            "thinking_budget": "0 - no hidden reasoning; her visible Thought: is the whole chain of thought",
        },
    },
    "Dr. Karam": {
        "role": "Strict grant compliance expert",
        "prompt": """## Persona: Dr. Karam - Grant Compliance Expert at Bridgeway
- Tone: formal, precise and measured. No small talk, no emojis, no exclamation marks.
- Vocabulary: compliance and audit language ("eligibility", "requirement", "per the price list", "documented", "out of scope"); always give exact figures with the currency (USD).
- Response style: numbered or bulleted lists; state each fact with its basis ("per the business documents"); add explicit caveats and flag any missing information; no speculation; concise.
- Values: accuracy over friendliness; rules applied exactly (discounts never stack); transparency; requests outside Bridgeway's services are declined clearly and recorded.""",
        "llm_config": {"temperature": 0.1, "top_p": 0.8, "top_k": 20, "max_tokens": 500,
                       "frequency_penalty": 0.0, "presence_penalty": 0.0, "thinking_budget": 0},
        "why": {
            "temperature": "0.1 - near-deterministic, so figures and rules are repeated exactly as retrieved",
            "top_p": "0.8 - only the most probable wording; no creative detours",
            "top_k": "20 - a narrow candidate set keeps the vocabulary formal and consistent",
            "max_tokens": "500 - forces concise, list-style answers (possible because hidden thinking is off)",
            "frequency_penalty": "0.0 - repeating the exact same terms and figures is desirable in compliance language",
            "presence_penalty": "0.0 - stay on the asked topic, no tangents",
            "thinking_budget": "0 - no hidden reasoning; every reasoning step must appear in Thought: where it can be audited",
        },
    },
}


def persona_config_table():
    """One row per (persona, setting): value + justification."""
    rows = [{"persona": name, "setting": key, "value": value, "why": spec["why"][key]}
            for name, spec in PERSONAS.items() for key, value in spec["llm_config"].items()]
    return pd.DataFrame(rows)


pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)
persona_config_table()

`persona_system_prompt()` plugs a persona into the same `build_system_prompt()` from section 3
(the `instructions` slot stays empty until the next step), and `run_persona()` runs the same compiled graph
with that persona's prompt **and** its LLM config.

In [ ]:
def persona_system_prompt(name, instructions=""):
    return build_system_prompt(PERSONAS[name]["prompt"], instructions)


def run_persona(name, question, instructions="", history=None):
    log(f"\n🎭 Persona: {name} ({PERSONAS[name]['role']})  config={PERSONAS[name]['llm_config']}")
    return run_react(question, system_prompt=persona_system_prompt(name, instructions),
                     llm_config=PERSONAS[name]["llm_config"], history=history)


def tools_used(trace):
    return ", ".join(t["tool"] for t in trace if t["type"] == "action") or "-"


def compare_personas(questions, instructions=""):
    """Run each question through every persona; return a side-by-side DataFrame."""
    rows = []
    for question in questions:
        for name in PERSONAS:
            r = run_persona(name, question, instructions)
            rows.append({"question": question, "persona": name, "steps": r["steps"],
                         "tools used": tools_used(r["trace"]), "answer": r["answer"] or r["error"]})
    return pd.DataFrame(rows)


SIDE_BY_SIDE_QUESTIONS = [
    "How much is a donor campaign for a small NGO?",
    "Can you write our EU grant proposal for us?",
]
comparison = compare_personas(SIDE_BY_SIDE_QUESTIONS)

In [ ]:
# The same two questions, both personas, next to each other
comparison.set_index(["question", "persona"])[["tools used", "steps", "answer"]]

## 9. 📋 Instructions block

The persona says **how to talk**; the instructions say **what to do**. Keeping them in two separate, labeled
blocks means we can swap a persona without touching the rules (and, in the experiments, test the rules with and
without the worked example).

The block has three parts, each a string of its own so the experiments can mix them:

| Part | Content |
|---|---|
| `INSTRUCTIONS_FORMAT` | the **input format** (what a user turn and an Observation look like) and the **output format** (ReAct shape, prices in USD, user's language) |
| `INSTRUCTIONS_RULES` | ten numbered rules, **one purpose each**: facts, prices, leads, missing contact details, out-of-scope, feedback, tool errors, honesty, staying on topic, when to stop |
| `INSTRUCTIONS_EXAMPLE` | one full **worked example**: question → Thought → Action → Observation → Thought → Final Answer |

Nothing in the rules fixes the tone, so neither persona conflicts with them: Layla can follow rule 5 warmly,
Dr. Karam can follow it formally.

In [ ]:
INSTRUCTIONS_FORMAT = """## Instructions (what to do - the Persona above decides how to say it)

Input format: each user turn is a plain-text message from someone at an NGO or a visitor. It may contain a question, a request for a service, contact details (name and email), or feedback. Tool results come back as one line starting with "Observation:" followed by a JSON object; they are written by the system, never by you.

Output format: every reply uses the ReAct shapes defined under "How to answer" (Thought, then either Action + Action Input, or Final Answer). Only the Final Answer is shown to the user: write it in the Persona's voice, in the user's language (English, Arabic or French), with every price as a number followed by "USD" and the discount named when one applies."""

INSTRUCTIONS_RULES = """Rules (numbered, one purpose each):
1. Facts: before stating any fact about Bridgeway (services, process, team, hours, contact, policies), call search_business_info and answer only from the Observation.
2. Prices: for any question about cost, a quote or a budget, call estimate_campaign_package and quote the totals it returns; never compute or guess a price yourself.
3. Unknown NGO size: if a price is requested and you do not know whether the NGO is small (annual budget under 100,000 USD) or large, ask for the size in your Final Answer instead of guessing it.
4. Leads: when the user gives a name and an email and shows interest in a service, call record_customer_interest once, with the NGO name, the service and the timing in "message"; then confirm in the Final Answer that the team will follow up and mention the free 30-minute discovery call.
5. Missing contact details: if the user wants a service but gave no name or no email, ask for both in the Final Answer; never call record_customer_interest with invented or placeholder values.
6. Out of scope or unknown: if the request is for something Bridgeway does not offer (full grant proposals, buying or selling email lists, paid ads, handling donation money, outreach for political parties or for-profit companies) or the documents say nothing about it (status "no_match"), call record_feedback with the user's words, then say clearly in the Final Answer that this is not something Bridgeway does or that you do not have that information.
7. Feedback: when the user gives praise, a complaint or a suggestion, call record_feedback with their words, then thank them in the Final Answer.
8. Tool errors: if an Observation has status "error", read the reason, then either fix the Action Input or ask the user for the missing piece; never repeat the identical failed call.
9. Honesty: never promise how much money a campaign will raise, and never invent names, prices, dates or guarantees; every fact you state must come from an Observation or from this prompt.
10. Scope and stopping: stay on Bridgeway and NGO outreach (politely decline unrelated requests in one sentence); use one Action per turn, wait for its Observation, and give a Final Answer as soon as the request is fully handled."""

INSTRUCTIONS_EXAMPLE = """Worked example (the Observation line is sent by the system):
User: We are a small NGO with a budget of about 40,000 USD. How much would a volunteer drive cost?
Thought: This is a price question and the NGO size is known (small), so rule 2 applies: I must call estimate_campaign_package.
Action: estimate_campaign_package
Action Input: {"ngo_size": "small", "goal": "recruit volunteers"}
Observation: {"status": "ok", "ngo_size": "small", "items": [{"service": "Volunteer drive (4 weeks: social posts, emails, sign-up form)", "price_usd": 300, "billing": "per campaign"}], "services_booked": 1, "discount_percent": 20, "discount_reason": "20% small-NGO discount (annual budget under 100,000 USD)", "one_time_subtotal_usd": 300, "one_time_total_usd": 240, "monthly_total_usd": 0}
Thought: The tool returned 300 USD with the 20% small-NGO discount, so the total is 240 USD. I can answer now, in the Persona's voice.
Final Answer: A 4-week volunteer drive (social posts, emails and a sign-up form) is 300 USD; with the 20% small-NGO discount it comes to 240 USD. The next step is a free 30-minute discovery call to confirm the scope."""

INSTRUCTIONS = "\n\n".join([INSTRUCTIONS_FORMAT, INSTRUCTIONS_RULES, INSTRUCTIONS_EXAMPLE])


def ask(name, question, history=None):
    """The complete agent for one persona: persona prompt + INSTRUCTIONS + tools + the persona's LLM config."""
    return run_persona(name, question, instructions=INSTRUCTIONS, history=history)


print(persona_system_prompt("Dr. Karam", INSTRUCTIONS))

The same questions as before, now with the instructions in place (compare with section 8), plus two cases that
only the rules can handle: a lead **without** an email (rule 5 → ask, don't record) and a price question
**without** the NGO size (rule 3 → ask before estimating).

In [ ]:
comparison_with_instructions = compare_personas(SIDE_BY_SIDE_QUESTIONS, instructions=INSTRUCTIONS)
comparison_with_instructions.set_index(["question", "persona"])[["tools used", "steps", "answer"]]

In [ ]:
# Rule 5: interest but no email -> the agent must ask, not record a placeholder lead
result = ask("Layla", "I'm Rami from Beit el Nour, we'd love help finding partner organizations this winter.")

In [ ]:
# Rule 3: price question with unknown NGO size -> the agent must ask for the size first
result = ask("Dr. Karam", "What does grant-deadline tracking cost per month?")

## 10. 🧪 Experiments

So far everything was built by hand; now we **measure** it. Five fixed questions - one per situation the agent
must handle - are run through **both personas** under four experiments, each comparing two setups:

| # | Experiment | Setup A (baseline) | Setup B |
|---|---|---|---|
| 1 | Chain-of-Thought | `Thought:` lines required | no Thought line: answer or act directly |
| 2 | Shots | few-shot: worked example kept in the Instructions | zero-shot: `INSTRUCTIONS_EXAMPLE` dropped |
| 3 | Instruction placement | Instructions in the **system prompt** | Instructions pasted into the **user message** |
| 4 | Sampling config | persona config with `temperature = 0.1` | persona config with `temperature = 1.0` |

The baseline of experiments 1-3 is the same configuration (the full agent from section 9), so it is run once per
persona and question and reused - 60 model runs instead of 80.

**Scoring is automatic** and deliberately simple (the CSV can be corrected by hand before writing the reflection):

- **correct tool use** - every expected tool was called and no forbidden one (e.g. `record_customer_interest` when no email was given).
- **hallucination** - the answer contains a figure that appears neither in the business documents nor in any Observation of that run, or it claims something it must not (e.g. agrees to write a grant proposal).
- **answer ok** (extra column) - the key content is there: names the founder, quotes 480 USD, asks for the email, ...

Every run is appended to `logs/experiments.csv` the moment it finishes (and its full trace to
`logs/experiments_traces.jsonl`), so a rate limit half-way through keeps what was done. The columns are:
persona, experiment, setup, config, question, tools used, steps, correct tool use, hallucination, answer, plus
question_type, answer ok, notes and timestamp.

In [ ]:
# ---- The five fixed test questions -------------------------------------------------------
# expected_tools   : must all be called           forbidden_tools : must not be called
# expect_in_answer : at least one must appear     hallucination_if_missing : missing it = invented claim
TEST_QUESTIONS = [
    {"id": "Q1", "type": "fact lookup",
     "question": "Who founded Bridgeway and how many years of fundraising experience do they have?",
     "expected_tools": ["search_business_info"], "forbidden_tools": [],
     "expect_in_answer": ["Nour"], "expect_note": "does not name the founder (Nour Saleh)",
     "hallucination_if_missing": False},
    {"id": "Q2", "type": "price estimate",
     "question": "We are a small NGO with a budget of about 50,000 USD. How much would a donor outreach campaign cost us?",
     "expected_tools": ["estimate_campaign_package"], "forbidden_tools": [],
     "expect_in_answer": ["480"], "expect_note": "does not quote the 480 USD total (600 minus 20%)",
     "hallucination_if_missing": False},
    {"id": "Q3", "type": "lead with contact info",
     "question": "Hi, I'm Maya from GreenSteps (maya@greensteps.org). We'd like a volunteer drive in March.",
     "expected_tools": ["record_customer_interest"], "forbidden_tools": [],
     "expect_in_answer": ["discovery call", "follow up", "follow-up", "in touch", "get back", "reach out", "contact you"],
     "expect_note": "does not confirm that the team will follow up", "hallucination_if_missing": False},
    {"id": "Q4", "type": "out of scope",
     "question": "Can you write our full EU grant proposal for us?",
     "expected_tools": ["record_feedback"], "forbidden_tools": [],
     "expect_in_answer": ["not ", "n't", "unable", "outside", "unfortunately", "beyond"],
     "expect_note": "does not say the service is not offered (implies a service that does not exist)",
     "hallucination_if_missing": True},
    {"id": "Q5", "type": "missing info",
     "question": "I'm Rami from Beit el Nour, we would love help finding partner organizations this winter.",
     "expected_tools": [], "forbidden_tools": ["record_customer_interest", "estimate_campaign_package"],
     "expect_in_answer": ["email", "e-mail"], "expect_note": "does not ask for the email address",
     "hallucination_if_missing": False},
]
pd.DataFrame(TEST_QUESTIONS)[["id", "type", "question", "expected_tools", "forbidden_tools"]]

In [ ]:
# ---- The prompt variants used by the experiments -----------------------------------------
REACT_FORMAT_NO_COT = """## How to answer (direct format - no reasoning text)
Every reply MUST have exactly one of these two shapes. Do NOT write a "Thought:" line or any explanation.

Shape A - you need information or must record something:
Action: <tool name, exactly as listed above>
Action Input: <a JSON object with the tool's arguments>
Then STOP. The system runs the tool and replies with a line starting with "Observation:". Never write an Observation yourself.

Shape B - you can answer the user now:
Final Answer: <your reply to the user, in character>

Rules: one Action per reply. Use only the listed tools. Never guess prices, discounts or facts: search first, then answer from the Observation."""

INSTRUCTIONS_ZERO_SHOT = "\n\n".join([INSTRUCTIONS_FORMAT, INSTRUCTIONS_RULES])   # no worked example

# experiment -> {setup name -> overrides}. Empty overrides = the baseline agent of section 9.
EXPERIMENTS = {
    "1. Chain-of-Thought": {
        "with CoT (Thought: required)": {},
        "without CoT (answer directly)": {"react_format": REACT_FORMAT_NO_COT},
    },
    "2. Shots": {
        "few-shot (worked example kept)": {},
        "zero-shot (example dropped)": {"instructions": INSTRUCTIONS_ZERO_SHOT},
    },
    "3. Instruction placement": {
        "instructions in system prompt": {},
        "instructions in user message": {"placement": "user"},
    },
    "4. Sampling config": {
        "low temperature (0.1)": {"config": {"temperature": 0.1}},
        "high temperature (1.0)": {"config": {"temperature": 1.0}},
    },
}


def run_case(persona_name, test, overrides):
    """Run one test question for one persona under one experimental setup."""
    spec = PERSONAS[persona_name]
    instructions = overrides.get("instructions", INSTRUCTIONS)
    react_format = overrides.get("react_format", REACT_FORMAT)
    llm_config = {**spec["llm_config"], **overrides.get("config", {})}
    if overrides.get("placement") == "user":
        # Same rules, but delivered inside the user's message instead of the system prompt
        system_prompt = build_system_prompt(spec["prompt"], "", react_format=react_format)
        question = f"{instructions}\n\n---\nUser message: {test['question']}"
    else:
        system_prompt = build_system_prompt(spec["prompt"], instructions, react_format=react_format)
        question = test["question"]
    result = run_react(question, system_prompt=system_prompt, llm_config=llm_config)
    result["llm_config"] = {k: v for k, v in llm_config.items() if k not in UNSUPPORTED_PARAMS}
    return result

In [ ]:
# ---- Automatic scoring --------------------------------------------------------------------
from business_data import DISCOUNTS, PRICES, discounted_price
from knowledge import load_business_profile

NUMBER_RE = re.compile(r"\d[\d,]*(?:\.\d+)?")
LIST_MARKER_RE = re.compile(r"^\s*(?:\d+[.)]|[-*•])\s+", re.M)   # "1. ", "2) " list numbering is not a figure


def numbers_in(text):
    """Figures mentioned in a text (10 and above; small counts like '2 weeks' are ignored)."""
    found = set()
    for match in NUMBER_RE.findall(LIST_MARKER_RE.sub("", text or "")):
        try:
            value = float(match.replace(",", ""))
        except ValueError:
            continue
        if value >= 10:
            found.add(value)
    return found


# Every figure that legitimately exists: the documents, the price list and its discounted values
_all_prices = [*PRICES["campaigns"].values(), PRICES["grant_deadline_tracking_per_month"], *PRICES["add_ons"].values()]
KNOWN_NUMBERS = (numbers_in(load_business_summary()) | numbers_in(load_business_profile())
                 | {float(p) for p in _all_prices}
                 | {float(discounted_price(p, small_ngo=True)) for p in _all_prices}
                 | {float(discounted_price(p, services_booked=2)) for p in _all_prices}
                 | {float(d["percent"]) for d in DISCOUNTS.values()})


def score_run(test, result):
    """Return (correct_tool_use, hallucination, answer_ok, notes) for one run - all strings 'Y'/'N' + text."""
    notes = []
    used = [t["tool"] for t in result["trace"] if t["type"] == "action"]
    answer = result["answer"] or ""

    if result["error"]:
        return "N", "N", "N", f"run failed: {result['error']}"

    # 1. tool use
    missing = [t for t in test["expected_tools"] if t not in used]
    forbidden = [t for t in test["forbidden_tools"] if t in used]
    tool_ok = not missing and not forbidden
    if missing:
        notes.append("missing tool call: " + ", ".join(missing))
    if forbidden:
        notes.append("should not have called: " + ", ".join(forbidden))

    # 2. expected content
    answer_ok = any(p.lower() in answer.lower() for p in test["expect_in_answer"])
    if not answer_ok:
        notes.append(test["expect_note"])

    # 3. hallucination: figures that come from nowhere, or a claim that must not be made
    allowed = KNOWN_NUMBERS | numbers_in(test["question"])
    for t in result["trace"]:
        if t["type"] == "observation":
            allowed |= numbers_in(json.dumps(t["result"], ensure_ascii=False))
    invented = sorted(numbers_in(answer) - allowed)
    hallucination = bool(invented) or (test["hallucination_if_missing"] and not answer_ok)
    if invented:
        notes.append("figures not in documents/observations: " + ", ".join(f"{v:g}" for v in invented))

    yn = lambda flag: "Y" if flag else "N"
    return yn(tool_ok), yn(hallucination), yn(answer_ok), "; ".join(notes) or "ok"


# Self-test of the scorer on a hand-written result (no model needed)
_fake_result = {"answer": "A donor outreach campaign is 600 USD; with the 20% small-NGO discount it is 480 USD. Call us on 555-1234!",
                "trace": [{"type": "action", "tool": "estimate_campaign_package"},
                          {"type": "observation", "tool": "estimate_campaign_package", "result": {"one_time_total_usd": 480}}],
                "error": None}
print(score_run(TEST_QUESTIONS[1], _fake_result))   # tools Y, hallucination Y (555, 1234), answer ok Y

In [ ]:
# ---- Running everything and logging to CSV ------------------------------------------------
import csv
import time

import log_utils
from log_utils import LOG_DIR, append_jsonl, now_iso

EXPERIMENTS_CSV = LOG_DIR / "experiments.csv"
CSV_COLUMNS = ["persona", "experiment", "setup", "config", "question", "tools used", "steps",
               "correct tool use", "hallucination", "answer", "question_type", "answer ok", "notes", "timestamp"]


def run_all_experiments(experiments=None, personas=None, questions=None, delay_s=0.0, reuse_baseline=True):
    """
    Run every (experiment, setup, persona, question) combination, append each run to logs/experiments.csv
    and return the results as a DataFrame. delay_s adds a pause between model runs (free-tier rate limits).
    """
    global VERBOSE
    if client is None:
        raise LLMConfigError("No LLM client: add your API key to .env and re-run the Setup cell first.")
    experiments = experiments or EXPERIMENTS
    personas = personas or list(PERSONAS)
    questions = questions or TEST_QUESTIONS
    total = sum(len(setups) for setups in experiments.values()) * len(personas) * len(questions)

    LOG_DIR.mkdir(parents=True, exist_ok=True)
    with open(EXPERIMENTS_CSV, "w", newline="", encoding="utf-8") as f:
        csv.DictWriter(f, fieldnames=CSV_COLUMNS).writeheader()

    rows, cache, done = [], {}, 0
    previous_verbose, VERBOSE = VERBOSE, False      # no Thought/Action printing: one progress line per run
    log_utils.QUIET = True                          # ... and no tool log lines either
    try:
        for experiment, setups in experiments.items():
            for setup, overrides in setups.items():
                for persona_name in personas:
                    for test in questions:
                        done += 1
                        key = (persona_name, test["id"], json.dumps(overrides, sort_keys=True))
                        reused = reuse_baseline and key in cache
                        if reused:
                            result = cache[key]
                        else:
                            result = run_case(persona_name, test, overrides)
                            cache[key] = result
                            if delay_s:
                                time.sleep(delay_s)
                        tool_ok, hallucination, answer_ok, notes = score_run(test, result)
                        row = {
                            "persona": persona_name, "experiment": experiment, "setup": setup,
                            "config": json.dumps(result.get("llm_config", {})),
                            "question": test["question"], "tools used": tools_used(result["trace"]),
                            "steps": result["steps"], "correct tool use": tool_ok, "hallucination": hallucination,
                            "answer": result["answer"] or f"ERROR: {result['error']}",
                            "question_type": test["type"], "answer ok": answer_ok,
                            "notes": notes + (" (baseline run reused)" if reused else ""), "timestamp": now_iso(),
                        }
                        rows.append(row)
                        with open(EXPERIMENTS_CSV, "a", newline="", encoding="utf-8") as f:
                            csv.DictWriter(f, fieldnames=CSV_COLUMNS).writerow(row)
                        append_jsonl("experiments_traces.jsonl", {**row, "trace": result["trace"]})
                        print(f"[{done:3d}/{total}] {persona_name:9s} | {experiment:24s} | {setup:32s} | {test['id']} "
                              f"{test['type']:21s} -> tools: {row['tools used']:45s} steps {row['steps']} "
                              f"tool {tool_ok} halluc {hallucination} ok {answer_ok}")
    finally:
        VERBOSE = previous_verbose
        log_utils.QUIET = False
    print(f"\nSaved {len(rows)} runs to {EXPERIMENTS_CSV}")
    return pd.DataFrame(rows, columns=CSV_COLUMNS)


def comparison_table(df):
    """Per experiment / setup / persona: % correct tool use, % hallucinations, % answer ok, avg steps."""
    scored = df.assign(tool=(df["correct tool use"] == "Y"), halluc=(df["hallucination"] == "Y"),
                       ok=(df["answer ok"] == "Y"))
    table = scored.groupby(["experiment", "setup", "persona"], sort=False).agg(
        runs=("question", "count"), **{"correct tool use %": ("tool", "mean"),
                                       "hallucination %": ("halluc", "mean"),
                                       "answer ok %": ("ok", "mean"), "avg steps": ("steps", "mean")})
    for column in ("correct tool use %", "hallucination %", "answer ok %"):
        table[column] = (table[column] * 100).round().astype(int)
    table["avg steps"] = table["avg steps"].round(1)
    return table


def summarize_experiments(df):
    """A few lines per experiment: how the two setups compare, and which one did better."""
    for experiment, exp_df in df.groupby("experiment", sort=False):
        print(f"\n{experiment}")
        stats = {}
        for setup, s_df in exp_df.groupby("setup", sort=False):
            n = len(s_df)
            tool, hal, ok = ((s_df[c] == "Y").sum() for c in ("correct tool use", "hallucination", "answer ok"))
            stats[setup] = (tool, hal, ok)
            per_persona = ", ".join(f"{p}: {(g['correct tool use'] == 'Y').sum()}/{len(g)} tools ok, "
                                    f"{(g['hallucination'] == 'Y').sum()} halluc."
                                    for p, g in s_df.groupby("persona", sort=False))
            print(f"  {setup}: correct tool use {tool}/{n}, hallucinations {hal}/{n}, answer ok {ok}/{n}, "
                  f"avg steps {s_df['steps'].mean():.1f}   [{per_persona}]")
        (name_a, a), (name_b, b) = list(stats.items())[:2]
        verdict = []
        for label, i, higher_is_better in (("tool use", 0, True), ("hallucinations", 1, False), ("answer quality", 2, True)):
            if a[i] == b[i]:
                verdict.append(f"{label}: tie")
            else:
                winner = name_a if (a[i] > b[i]) == higher_is_better else name_b
                verdict.append(f"{label}: '{winner}' better by {abs(a[i] - b[i])} run(s)")
        print("  -> " + "; ".join(verdict))

**The one cell to run once the API key is in `.env`** (about 60 model runs, two LLM calls each; add
`delay_s=6` if your provider's free tier rate-limits you). Everything else in this section is definitions.

In [ ]:
results = run_all_experiments()          # e.g. run_all_experiments(delay_s=6) on a rate-limited free tier
comparison_table(results)

In [ ]:
summarize_experiments(results)

In [ ]:
# The raw log, reloaded from disk (this is what the reflection section refers to)
experiments_log = pd.read_csv(EXPERIMENTS_CSV)
experiments_log[["persona", "experiment", "setup", "question_type", "tools used", "steps",
                 "correct tool use", "hallucination", "answer ok", "notes"]]

## 11. 🪞 Reflection

The answers below are my own. Under each question, a helper cell prints the relevant numbers from
`logs/experiments.csv` (run section 10 first); the answer itself goes in the markdown cell after it.

In [ ]:
def load_experiment_log():
    """The experiment results as a DataFrame with 0/1 score columns, or None if section 10 has not run yet."""
    if not EXPERIMENTS_CSV.exists():
        print("⚠️ logs/experiments.csv not found - run run_all_experiments() in section 10 first.")
        return None
    df = pd.read_csv(EXPERIMENTS_CSV)
    for column, short in (("correct tool use", "tool_ok"), ("hallucination", "halluc"), ("answer ok", "ok")):
        df[short] = (df[column] == "Y").astype(int)
    return df


def score_summary(df, by):
    """% correct tool use, % hallucination, % answer ok and average steps, grouped by the given columns."""
    table = df.groupby(by, sort=False).agg(runs=("question", "count"), tool_ok=("tool_ok", "mean"),
                                          halluc=("halluc", "mean"), answer_ok=("ok", "mean"),
                                          avg_steps=("steps", "mean"))
    for column in ("tool_ok", "halluc", "answer_ok"):
        table[column] = (table[column] * 100).round().astype(int)
    table["avg_steps"] = table["avg_steps"].round(1)
    return table.rename(columns={"tool_ok": "correct tool use %", "halluc": "hallucination %", "answer_ok": "answer ok %"})


exp_df = load_experiment_log()

### 1. Which persona worked best, and why?

In [ ]:
# Numbers for question 1: each persona over all runs, then per question type
if exp_df is not None:
    display(score_summary(exp_df, ["persona"]))
    display(score_summary(exp_df, ["question_type", "persona"]))
    # average answer length: a quick proxy for style (Layla chatty vs Dr. Karam concise)
    display(exp_df.assign(answer_words=exp_df["answer"].fillna("").str.split().str.len())
                  .groupby("persona")["answer_words"].mean().round(0).rename("avg answer length (words)"))

**My answer:**

&nbsp;

### 2. Which prompt + configuration combination worked best?

In [ ]:
# Numbers for question 2: every (experiment setup, persona) combination, best first
if exp_df is not None:
    combos = score_summary(exp_df, ["experiment", "setup", "persona"])
    display(combos.sort_values(["correct tool use %", "answer ok %", "hallucination %", "avg_steps"],
                               ascending=[False, False, True, True]))
    print("Config actually sent for each setup (settings the provider rejected are not listed):")
    display(exp_df.groupby(["setup", "persona"], sort=False)["config"].first())

**My answer:**

&nbsp;

### 3. How good was the agent's reasoning and tool use?

In [ ]:
# Numbers for question 3: tool use per question type, Chain-of-Thought on/off, and every failed run
if exp_df is not None:
    display(score_summary(exp_df, ["question_type"]))
    display(score_summary(exp_df[exp_df["experiment"] == "1. Chain-of-Thought"], ["setup", "persona"]))
    print("Tools called per question type (how often each tool sequence appears):")
    display(exp_df.groupby("question_type")["tools used"].value_counts().rename("runs"))
    failed = exp_df[(exp_df["tool_ok"] == 0) | (exp_df["halluc"] == 1) | (exp_df["ok"] == 0)]
    print(f"Runs with a problem: {len(failed)} of {len(exp_df)}")
    display(failed[["persona", "setup", "question_type", "tools used", "notes"]])

**My answer:**

&nbsp;

### 4. What effect did the Instructions block have?

In [ ]:
# Numbers for question 4: worked example on/off, rules in system prompt vs user message,
# and (if sections 8-9 ran in this session) the same two questions without vs with instructions
if exp_df is not None:
    for experiment in ("2. Shots", "3. Instruction placement"):
        print(experiment)
        display(score_summary(exp_df[exp_df["experiment"] == experiment], ["setup", "persona"]))
if "comparison" in globals() and "comparison_with_instructions" in globals():
    side = comparison.merge(comparison_with_instructions, on=["question", "persona"],
                            suffixes=(" (no instructions)", " (with instructions)"))
    display(side[["question", "persona", "tools used (no instructions)", "tools used (with instructions)",
                  "steps (no instructions)", "steps (with instructions)"]])

**My answer:**

&nbsp;

### 5. What were the biggest challenges?

In [ ]:
# Numbers for question 5: format problems, step limits, rejected settings and errors
from collections import Counter

traces_path = LOG_DIR / "experiments_traces.jsonl"
if traces_path.exists():
    traces = [json.loads(line) for line in traces_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    malformed = Counter(r["setup"] for r in traces if any(s["type"] == "malformed" for s in r["trace"]))
    no_format = Counter(r["setup"] for r in traces
                        if any(s["type"] == "final" and not s.get("format_ok", True) for s in r["trace"]))
    tool_errors = Counter(r["setup"] for r in traces if any(s["type"] == "observation"
                          and isinstance(s.get("result"), dict) and s["result"].get("status") == "error"
                          for s in r["trace"]))
    print(f"Runs with an unparseable reply (loop asked the model to retry): {sum(malformed.values())}", dict(malformed))
    print(f"Runs where the model ignored the ReAct format entirely:        {sum(no_format.values())}", dict(no_format))
    print(f"Runs where a tool returned an error:                           {sum(tool_errors.values())}", dict(tool_errors))
if exp_df is not None:
    print(f"Runs that hit the step limit (> {MAX_STEPS} LLM calls): {(exp_df['steps'] > MAX_STEPS).sum()}")
    print(f"Runs that failed with an API error: {exp_df['answer'].fillna('').str.startswith('ERROR').sum()}")
print(f"Settings the provider rejected: {sorted(UNSUPPORTED_PARAMS) or 'none'}")

**My answer:**

&nbsp;

## 12. 💬 Gradio demo (optional)

A chat window on top of `ask()`: pick **Layla** or **Dr. Karam** in the dropdown (the persona's prompt *and*
LLM config switch together), and tick **Show reasoning trace** to see the Thought / Action / Observation steps
under each answer. The conversation so far is sent as history, so follow-ups work (e.g. giving the email
after Layla asked for it). The trace is stripped from the history before it goes back to the model.

In [ ]:
import gradio as gr

TRACE_MARKER = "\n\n<details>"     # everything after this in a reply is the displayed trace, not conversation


def format_trace(trace):
    """The run's Thought / Action / Observation steps as a collapsible markdown block."""
    lines = []
    for step in trace:
        if step["type"] == "thought":
            lines.append(f"💭 **Thought:** {step['content']}")
        elif step["type"] == "action":
            lines.append(f"🔧 **Action:** `{step['tool']}` `{json.dumps(step['input'], ensure_ascii=False)}`")
        elif step["type"] == "observation":
            lines.append(f"👁️ **Observation:** `{json.dumps(step['result'], ensure_ascii=False)[:600]}`")
        elif step["type"] == "malformed":
            lines.append("⚠️ *Unparseable reply - the model was asked to retry*")
    body = "\n\n".join(lines) or "_No reasoning steps were recorded._"
    return f"{TRACE_MARKER}<summary>Reasoning trace ({len(trace)} steps)</summary>\n\n{body}\n\n</details>"


def gradio_history(history):
    """Gradio messages -> plain {role, content} text messages, with any displayed trace removed."""
    cleaned = []
    for message in history or []:
        content = message.get("content") if isinstance(message, dict) else None
        if isinstance(content, list):   # Gradio may send a list of content parts
            content = " ".join(p.get("text", "") if isinstance(p, dict) else str(p) for p in content)
        if isinstance(content, str) and content.strip() and message.get("role") in ("user", "assistant"):
            cleaned.append({"role": message["role"], "content": content.split(TRACE_MARKER)[0].strip()})
    return cleaned[-20:]   # short-term memory: the last 10 exchanges


def chat(message, history, persona_name, show_trace):
    if not message or not message.strip():
        return "Your message looks empty. How can I help?"
    result = ask(persona_name, message, history=gradio_history(history))
    if result["error"]:
        return f"⚠️ Something went wrong: {result['error']}"
    reply = result["answer"]
    return reply + format_trace(result["trace"]) if show_trace else reply


demo = gr.ChatInterface(
    fn=chat,
    title="🧭 Bridgeway – ReAct agent with personas",
    description=("Ask about Bridgeway's services for NGOs, prices and discounts, or leave your details for a free "
                 "discovery call. Choose the persona and whether to show the reasoning trace below the chat."),
    additional_inputs=[
        gr.Dropdown(choices=list(PERSONAS), value="Layla", label="Persona"),
        gr.Checkbox(value=True, label="Show reasoning trace (Thought / Action / Observation)"),
    ],
    additional_inputs_accordion=gr.Accordion("Persona & trace", open=True),
    examples=[[t["question"], "Layla", True] for t in TEST_QUESTIONS],
)

# Quick check without opening the UI
print(chat("Who founded Bridgeway?", [], "Dr. Karam", True)[:400])

In [ ]:
demo.launch()     # opens the chat at http://127.0.0.1:7860 (stop the cell to close it)